**Phase 1**



• ✅ Task 1.1: Created a GitHub repository and uploaded project files.

• ✅ Task 1.2: Downloaded the arabic-generated-abstracts dataset using the Hugging Face datasets library.

• ✅ Task 1.3: Loaded and explored the dataset (checked columns, data types, label distribution, and data quality).



✅ Phase 1 completed successfully.

In [ ]:
from datasets import load_dataset
import pandas as pd
# Load the dataset from Hugging Face
dataset = load_dataset("KFUPM-JRCAI/arabic-generated-abstracts")

rows = []
for subset_name, subset in dataset.items():
    df = subset.to_pandas()

    #Human-written texts
    for text in df["original_abstract"]:
        rows.append({"text": text, "label": 0, "subset": subset_name, "model": "human"})

 # Machine-generated texts
    model_map = {
        "allam_generated_abstract": "allam",
        "jais_generated_abstract": "jais",
        "llama_generated_abstract": "llama",
        "openai_generated_abstract": "openai",
    }
    for col, mname in model_map.items():
        for text in df[col]:
            rows.append({"text": text, "label": 1, "subset": subset_name, "model": mname})

combined_df = pd.DataFrame(rows, columns=["text", "label", "subset", "model"])

print("✅ Combined dataframe created successfully!")
print("الشكل:", combined_df.shape)


In [ ]:
# ======================================================
#  Basic Information
# ======================================================

print("◆ Number of rows:", combined_df.shape[0])
print("◆ Number of columns:", combined_df.shape[1])

print("\n Column names:")
print(list(combined_df.columns))

print("\n Data types:")
print(combined_df.dtypes)

# ======================================================
#   Missing Values
# ======================================================

missing_count = combined_df.isna().sum()
missing_percent = (missing_count / len(combined_df) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
      'Missing %': missing_percent
})
print("\n Missing Values:")
print(missing_summary)

# ======================================================
#  Duplicates
# ======================================================


# Duplicated rows (full match across all columns)
duplicate_rows = combined_df.duplicated().sum()
print("\n Number of duplicated rows (full match):", duplicate_rows)

# Duplicated text only
duplicate_texts = combined_df['text'].duplicated().sum()
print(" Number of duplicated texts in 'text' column:", duplicate_texts)

# ======================================================
#  Label Distribution
# ======================================================

print("\n Label distribution (0=Human, 1=AI):")
print(combined_df['label'].value_counts())

print("\n Distribution by subset:")
print(combined_df['subset'].value_counts())

print("\n Distribution by model:")
print(combined_df['model'].value_counts())


In [ ]:

print("🔹 عدد الصفوف:", combined_df.shape[0])
print("🔹 عدد الأعمدة:", combined_df.shape[1])

print("\n📋 أسماء الأعمدة:")
print(list(combined_df.columns))

print("\n⚙️ أنواع البيانات:")
print(combined_df.dtypes)

In [ ]:

missing_count = combined_df.isna().sum()
missing_percent = (missing_count / len(combined_df) * 100).round(2)
missing_summary = pd.DataFrame({
    'عدد القيم المفقودة': missing_count,
    'النسبة المئوية %': missing_percent
})
print("\n🕳️ القيم المفقودة:")
print(missing_summary)

In [ ]:
duplicate_rows = combined_df.duplicated().sum()
print("\n🔁 عدد الصفوف المكررة بالكامل:", duplicate_rows)

# عدد النصوص المكررة في عمود text فقط
duplicate_texts = combined_df['text'].duplicated().sum()
print("🔁 عدد النصوص المكررة في عمود text:", duplicate_texts)


In [ ]:
print("\n📊 توزيع القيم في label (0=human, 1=AI):")
print(combined_df['label'].value_counts())

print("\n📊 توزيع حسب subset:")
print(combined_df['subset'].value_counts())

print("\n📊 توزيع حسب model:")
print(combined_df['model'].value_counts())

In [ ]:
# =========================================================
#  Display subsets (without count/unique) and show only text columns
# =========================================================

from IPython.display import display
import pandas as pd


for subset_name, subset in dataset.items():
    print(f"\n===== {subset_name.upper()} =====\n")

    ## Convert subset to DataFrame
    df = subset.to_pandas()

    display(df.head(5))


In [ ]:
# =========================================================
#  Display only columns and text for by_polishing subset
# =========================================================

from IPython.display import display

# Convert the selected subset to a DataFrame
df_polishing = dataset["by_polishing"].to_pandas()


display(df_polishing.head(5))


In [ ]:
#  Display all columns in the by_polishing subset including split info
df = dataset["by_polishing"].to_pandas()

# # Show column names
print(list(df.columns))

# # Display the first 5 rows
from IPython.display import display
display(df.head(5))


display(df.describe(include='object').T[['count', 'unique', 'top', 'freq']])


In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df = pd.DataFrame(dataset['by_polishing'])

#  Human texts
df_human = pd.DataFrame({
    'text': df['original_abstract'],
    'label': 'human',
    'source': 'original_abstract',
    'split': 'by_polishing'
})

# AI texts (example from JAIS model)
df_ai = pd.DataFrame({
    'text': df['jais_generated_abstract'],
    'label': 'ai',
    'source': 'jais_generated_abstract',
    'split': 'by_polishing'
})
# Combine both DataFrames
combined = pd.concat([df_human, df_ai], ignore_index=True)


# عرض أول 5 صفوف
combined.head(5)

In [ ]:
combined.to_csv("combined_data.csv", index=False)

In [ ]:
combined.columns

In [ ]:
dataset = load_dataset("KFUPM-JRCAI/arabic-generated-abstracts")

# Convert each subset into long format (Human / AI)
def make_long(ds):
    df= ds.to_pandas()[[
        "original_abstract",
        "allam_generated_abstract",
        "jais_generated_abstract",
        "llama_generated_abstract",
        "openai_generated_abstract"
    ]]
    long = df.melt(value_vars=df.columns, var_name="src", value_name="text")
    long["label"] = long["src"].apply(lambda x: "Human" if x == "original_abstract" else "AI")
    return long[["text", "label"]]

#  Combine the three subsets
df_full = pd.concat([
    make_long(dataset["by_polishing"]),
    make_long(dataset["from_title"]),
    make_long(dataset["from_title_and_content"])
], ignore_index=True)

#  Clean the texts
df_full["text"] = df_full["text"].astype(str).str.strip()
df_full = df_full.dropna(subset=["text"])
df_full = df_full[df_full["text"] != ""]
#  Preview and label distribution
print("Sample of the first 10 rows:\n")
print(df_full.head(10))

print("\n Label distribution (Human / AI):")
print(df_full["label"].value_counts())


**Phase 2:**

• ✅ Task 2.1: Cleaned the dataset and applied Arabic text preprocessing, including normalization, diacritics removal, stopword removal, and stemming.

In [ ]:
import re
import pandas as pd

# ========= Normalization helpers =========
_ar_diac   = re.compile(r"[\u0617-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")  # التشكيل
_tatweel   = re.compile(r"\u0640")                                            # التطويل

def _normalize_ar(s: str) -> str:
    s = _ar_diac.sub("", s)                 # Remove diacritics
    s = _tatweel.sub("", s)                 # Remove tatweel
    s = re.sub("[إأآا]", "ا", s)            # Normalize Alif
    s = re.sub("[ى]", "ي", s)               # Normalize yaa
    return s

# f30 keeps only the question mark punctuation
KEEP_PUNCT = "؟"

def clean_ar_text(text: str, *, remove_stopwords: bool = False) -> str:
    if not isinstance(text, str):
        text = str(text)

    t = text

    # Remove links, mentions, hashtags
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"[@#]\w+", " ", t)

    # Remove emails + numbers
    t = re.sub(r"\S+@\S+\s*", " ", t)
    t = re.sub(r"\d+", " ", t)


    t = _normalize_ar(t)

   # Important: keep English letters + Arabic letters + the question mark "؟"
    # Example: keep abbreviations like U.S. or Ph.D
    allowed = fr"A-Za-z\u0600-\u06FF\.\s{re.escape(KEEP_PUNCT)}"
    t = re.sub(fr"[^{allowed}]", " ", t)

    # spaces
    t = re.sub(r"\s+", " ", t).strip()
    return t


df_full["text_clean"] = df_full["text"].apply(lambda x: clean_ar_text(x, remove_stopwords=False))

print("\n Sample of texts before and after cleaning:\n")
df_full.head()

# ====== 5️ Save cleaned data======



✅ Task 2.2: Performed EDA and created visualizations, including word clouds, frequency distributions, and statistical analysis to compare patterns between human and AI-generated texts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Calculate average word length for each text
df_full["avg_word_len"] = df_full["text_clean"].apply(
    lambda x: np.mean([len(w) for w in str(x).split()]) if isinstance(x, str) and x.strip() else 0
)

#Calculate overall average word length by class (Human / AI)
avg_by_class = df_full.groupby("label")["avg_word_len"].mean()

# Plot the bar chart
plt.bar(avg_by_class.index, avg_by_class.values, color=["skyblue", "orange"])
plt.title("Average Word Length by Class")
plt.xlabel("Class")
plt.ylabel("Average Word Length")
plt.show()

In [ ]:
import re, numpy as np, seaborn as sns, matplotlib.pyplot as plt


TEXT_COL = "text_clean" if "text_clean" in df.columns else "text"

#Calculate average sentence length (in words)
splitter = re.compile(r"[\.!\?؟…؛]+")
df_full["avg_sent_len"] = df_full[TEXT_COL].astype(str).apply(
    lambda x: np.mean([len(s.split()) for s in splitter.split(x) if s.strip()]) if x.strip() else 0
)

# Filter outliers ( for better visualization)
filtered = df_full[df_full["avg_sent_len"] < 400]

# Plot the violin plot
sns.violinplot(x="label", y="avg_sent_len", data=filtered, palette=["skyblue","salmon"])
plt.title("Violin Plot of Sentence Lengths (Filtered) - Human vs AI")
plt.xlabel("Text Class"); plt.ylabel("Average Sentence Length (words)")
plt.show()

In [ ]:
import os, pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from matplotlib import font_manager



#  Prepare text for each class
ai_text    = " ".join(df_full[df_full["label"]=="AI"]["text"].dropna().astype(str))
human_text = " ".join(df_full[df_full["label"]=="Human"]["text"].dropna().astype(str))

# Find or set a suitable Arabic font for Matplotlib
font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
if not os.path.exists(font_path):
    font_path = font_manager.findfont("DejaVu Sans")  # يرجّع مسار ملف خط متاح

#  Generate and display word clouds (AI vs Human)
for txt, title, cmap in [
    (ai_text, "Arabic Word Cloud - ai", "winter"),
    (human_text, "Arabic Word Cloud - human", "summer")
]:
    wc = WordCloud(
        font_path=font_path,
        width=900, height=600,
        background_color="white",
        max_words=300,
        colormap=cmap,
        collocations=False
    ).generate(txt)

    plt.figure(figsize=(8,6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=16)
    plt.show()

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer



#  Function to get top n-grams
def top_ngrams(texts, ngram_range=(1,1), top_k=30, min_df=5):
    vec = CountVectorizer(
        ngram_range=ngram_range,
        token_pattern=r'(?u)\b[\u0621-\u064A]+\b',  # Arabic words only
        min_df=min_df
    )
    X = vec.fit_transform(texts.astype(str))
    freqs = np.asarray(X.sum(axis=0)).ravel()
    vocab = np.array(vec.get_feature_names_out())
    idx = np.argsort(freqs)[::-1][:top_k]
    return pd.DataFrame({"ngram": vocab[idx], "freq": freqs[idx]})

#Split texts by label (Human / AI)
human_texts = df_full.loc[df_full["label"]=="Human", "text"]
ai_texts    = df_full.loc[df_full["label"]=="AI",    "text"]

# Generate top unigrams and bigrams
top_uni_h = top_ngrams(human_texts, (1,1), top_k=30, min_df=5)  # Unigrams Human
top_uni_a = top_ngrams(ai_texts,    (1,1), top_k=30, min_df=5)  # Unigrams AI
top_bi_h  = top_ngrams(human_texts, (2,2), top_k=30, min_df=5)  # Bigrams Human
top_bi_a  = top_ngrams(ai_texts,    (2,2), top_k=30, min_df=5)  # Bigrams AI

#Display top 15 n-grams from each table
print("Top unigrams (Human)"); display(top_uni_h.head(15))
print("Top unigrams (AI)");    display(top_uni_a.head(15))
print("Top bigrams (Human)");  display(top_bi_h.head(15))
print("Top bigrams (AI)");     display(top_bi_a.head(15))

**Phase 3**



The features I am responsible for are:

9. Total number of words (N)✔️

30. Number of question marks✔️

51. Number of abbreviations ✔️

72. Grammatical person features: 3rd person✔️

93. Emotional arousal score✔️

In [ ]:

import re
import pandas as pd

def words_basic(s: str):
    """تُرجع قائمة الكلمات العربية والإنجليزية في النص"""
    return re.findall(r'\w+', str(s))

# ======(Feature 9) Total number of words======
def f9_total_words(s: str) -> int:
    return len(words_basic(s))


# ===== (Feature 30) Number of question marks====
def f30_num_question_marks(s: str) -> int:
    return len(re.findall(r'[؟?]', str(s)))


# (Feature 51) Number of abbreviations (Arabic/English)
_abbrev_pattern = re.compile(
    r'(\b[A-Za-z]{1,3}\.)|'       # مثل Dr. أو U.S.
    r'(\b[A-Za-z]{1,4}/[A-Za-z]{1,4})|'  # مثل Ph.D / M.Sc
    r'(\b[اأإآء-ي]{1,2}\.)'       # مثل د. م.
)

def f51_num_abbreviations(s: str) -> int:
    return len(_abbrev_pattern.findall(str(s)))




# ===(f72) Third-person feature======
third_pronouns = {"هو", "هي", "هم", "هما", "هن", "له", "لها", "فيها", "عليه", "عليهم", "عنهم"}

def f72_third_person_feature(s: str) -> float:
    ws = words_basic(s)
    if not ws:
        return 0.0
    pron_count = sum(1 for w in ws if w in third_pronouns)
    verb3_count = sum(1 for w in ws if w.startswith("ي") and len(w) >= 3)
    return (pron_count + verb3_count) / len(ws)


# ===== (f93) Emotional arousal score======
arousal_words = {
    "سعيد","غاضب","محبط","مروع","كارثة","كارثي","غاضبة",
    "متوتر","قلق","صادم","مثير","مفزع","ممتع","مأساة",
    "محزن","مفاجئ","مندهش","شغوف","متحمس"
}

def f93_emotional_arousal_score(text_base: str, text_orig: str) -> float:
    ws = words_basic(text_base)
    if not ws:
        return 0.0
    aw = sum(1 for w in ws if w in arousal_words)
    bangs = len(re.findall(r'[!！]', str(text_orig)))
    return (aw + bangs) / len(ws)




df_full["f9_total_words"] = df_full["text_clean"].apply(f9_total_words)
df_full["f30_qmarks"] = df_full["text_clean"].apply(f30_num_question_marks)
df_full["f51_abbrev"] = df_full["text_clean"].apply(f51_num_abbreviations)
df_full["f72_third_person_ratio"] = df_full["text_clean"].apply(f72_third_person_feature)
df_full["f93_emotion_score"] = df_full.apply(lambda x: f93_emotional_arousal_score(x["text_clean"], x["text_clean"]), axis=1)


preview_cols = ["text_clean","label","f9_total_words","f30_qmarks","f51_abbrev","f72_third_person_ratio","f93_emotion_score"]
df_full[preview_cols].head(10)

In [ ]:
combined.to_csv("combined_data_final.csv", index=False)

TASK 4.1
in this task, I trained a simple baseline model using Logistic Regression with TF-IDF features. The model achieved an accuracy of 0.96, which serves as the benchmark for later models

In [ ]:
# ===========================
# Task 4.1: Baseline Model (TF-IDF + Logistic Regression)
# ===========================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelBinarizer
import joblib, os

# Load the dataset


print("Data loaded successfully")
print("Shape:", df_full.shape)
print(df_full.head(3))

#Define features and labels
X = df_full["text_clean"].astype(str)
y = df_full["label"].astype(str)

# Split the dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#  Convert text to numeric representation using TF-IDF
tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),   # unigram + bigram
    min_df=3,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Create and train the baseline model (Logistic Regression)
model = LogisticRegression(max_iter=300, class_weight="balanced")
model.fit(X_train_tfidf, y_train)
# Train the model
model.fit(X_train_tfidf, y_train)

#Evaluate on test set
y_pred = model.predict(X_test_tfidf)


print("\n Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Compute ROC-AUC (Human vs AI)
lb = LabelBinarizer()
y_test_bin = lb.fit_transform(y_test)
y_proba = model.predict_proba(X_test_tfidf)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test_bin, y_proba))

#  حفظ النموذج والـ TF-IDF
# os.makedirs("models", exist_ok=True)
# joblib.dump(model, "models/baseline_logreg.joblib")
# joblib.dump(tfidf, "models/tfidf_vectorizer.joblib")
# print("\n Model and vectorizer saved inside 'models/' folder!")

In [ ]:
# 1) Check available columns
print(df_full.columns.tolist())

# 2) Define the feature and label columns you actually have
feature_cols = [
    "f9_total_words",
    "f30_qmarks",
    "f51_abbrev",
    "f72_third_person_ratio",
    "f93_emotion_score"
]


feature_cols = [c for c in feature_cols if c in df_full.columns]

# 3) Create feat_df with only the available feature columns and the label column
feat_df = df_full[feature_cols + ["label"]].copy()



TASK 4.2

For this task, I implemented and tuned two additional classifiers — Support Vector Machine (SVM) and Random Forest — using the extracted feature sets.
I also wrote a code that automatically compares the models based on their accuracy and identifies which one performs better.
Additionally, I plotted the confusion matrix for both models to visualize their classification performance.
These experiments provided a deeper understanding of how traditional models perform on the dataset.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


X = feat_df.drop(columns=["label"]).values
y = feat_df["label"].values  # Human / AI

# Test 20% - Train 80%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 1) SVM
svm = LinearSVC(random_state=42)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

print("=== [SVM] تقييم ===")
print(classification_report(y_test, svm_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, svm_pred))
print("Accuracy:", round(accuracy_score(y_test, svm_pred), 3))

# 2) Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("\n=== [RandomForest] تقييم ===")
print(classification_report(y_test, rf_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))
print("Accuracy:", round(accuracy_score(y_test, rf_pred), 3))

# إظهار أهمية الميزات من الراندم فوريست
import numpy as np
feature_names = ["f9_total_words","f30_qmarks","f51_abbrev","f72_third","f93_arousal"]
importances = rf.feature_importances_
for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
   print(f"{name:>15}: {imp:.3f}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Calculate performance for each model
svm_acc = accuracy_score(y_test, svm_pred)
rf_acc = accuracy_score(y_test, rf_pred)

svm_f1 = f1_score(y_test, svm_pred, average='weighted')
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

# Determine which model performs better based on Accuracy
if svm_acc > rf_acc:
    print(f" Best model SVM  |  Accuracy = {svm_acc:.2f}")
else:
    print(f" Best model Random Forest  |  Accuracy = {rf_acc:.2f}")


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# SVM
ConfusionMatrixDisplay.from_predictions(y_test, svm_pred,
                                        display_labels=["Human", "AI"],
                                        cmap="Blues")
plt.title("SVM - Confusion Matrix")
plt.show()

# Random Forest
ConfusionMatrixDisplay.from_predictions(y_test, rf_pred,
                                        display_labels=["Human", "AI"],
                                        cmap="Greens")
plt.title("Random Forest - Confusion Matrix")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select only my feature columns
features = ["f9_total_words", "f30_qmarks", "f51_abbrev", "f72_third_person_ratio", "f93_emotion_score"]

# Compute the correlation matrix
corr = df_full[features].corr()

# Plot it as a heatmap
plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True, cmap="Purples", fmt=".2f")
plt.title("Feature Correlation Matrix (My 5 Features)")
plt.show()

Task 4.3 – Deep Learning Model (AraBERT)

In the final task, I built a deep learning model using AraBERT embeddings and a simple feedforward neural network for classification.
This model achieved the highest accuracy of 0.98, outperforming all previous models from Tasks 4.1 and 4.2.
Therefore, the AraBERT-based model was selected as the best-performing approach in this project.

In [ ]:
!pip -q install transformers torch torchvision torchaudio scikit-learn tqdm


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm


# Select the correct column for text
TEXT_COL = "text_clean"

df_full= df_full[[TEXT_COL, "label"]].dropna()
df_full[TEXT_COL] = df_full[TEXT_COL].astype(str)

# Encode the labels
label2id = {"Human": 0, "AI": 1}
id2label = {v:k for k,v in label2id.items()}
df_full["target"] = df_full["label"].map(label2id)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    df_full[TEXT_COL].values, df_full["target"].values, test_size=0.2, random_state=42, stratify=df_full["target"]
)


In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv2"
device = torch.device("cuda:0")  # نشتغل على الـGPU
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert.eval()

@torch.no_grad()
def get_embeddings(texts, batch_size=16, max_len=128):
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = list(texts[i:i+batch_size])
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        out = bert(**enc)
        cls_vec = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(cls_vec)
    return np.vstack(all_vecs)

# استخراج التمثيلات العددية
X_train_vec = get_embeddings(X_train)
X_test_vec = get_embeddings(X_test)



In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, in_dim=768, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 2)
        )
    def forward(self, x):
        return self.net(x)

model = SimpleNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
def to_tensor(x):
    return torch.tensor(x, dtype=torch.float32).to(device)

X_train_t = to_tensor(X_train_vec)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t  = to_tensor(X_test_vec)
y_test_t  = torch.tensor(y_test, dtype=torch.long).to(device)

EPOCHS = 5
BATCH = 32

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(len(X_train_t))
    total_loss = 0
    for i in range(0, len(X_train_t), BATCH):
        idx = perm[i:i+BATCH]
        xb, yb = X_train_t[idx], y_train_t[idx]
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss:.3f}")


In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test_t)
    preds = torch.argmax(preds, dim=1).cpu().numpy()

print("\n=== model results===")
print(classification_report(y_test, preds, target_names=["Human", "AI"], digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))